# 05. HPO — ハイパーパラメータ探索

README の工程 **④ HPO** にあたる。実行は `src/05_hpo.py`。

## なぜ HPO を最後に回したのか

前コンペ S6E8 で「スコアが伸び悩むとすぐ Optuna に逃げてしまい、実際には効かなかった」という反省が
あり、`CLAUDE.md` で **HPO は特徴量エンジニアリングをやり切った後**と決めていた。

実際その判断は正しかった。ここまでの改善の内訳は次のとおり。

| 施策 | 改善幅 | 種別 |
|---|---|---|
| 厳密値 Target Encoding | +0.003 | FE |
| 収束の確認(lr 引き下げ + early stopping) | +0.0008 | パラメータ |
| Triple TE + digit + ビン数1024 | +0.0005〜0.001 | FE |
| 列サブサンプリング(`colsample=0.3`, `max_depth=5`) | +0.0002 | パラメータ |

FE が桁違いに効いており、パラメータ調整は後から効いた。ただし**列サブサンプリングのように
「見落としていた既定値」は例外的に大きい**。これは探索というより是正だった。

## 現在地と期待値

- 最良: CV 0.94623 / **Public LB 0.94645**(273位 / 2543チーム、上位10.7%)
- 目標(上位15%)は達成済み。1位は 0.94675 で、差は 0.00030

**期待値は低いと見ている。** `research.md` の外部調査によれば、列サブサンプリング導入後に
深さや列比率をさらに振った試行(`d4` / `d6` / `ff02` / `ff04` / `mc30` / `lr01`)は
**すべて -0.000023〜+0.000017** で、誤差の範囲だった。

それでも回す理由は2つ。

1. **LightGBM の `num_leaves` がデフォルトの 31 のまま**で、特徴量が 13 → 92 列に増えている。
   ここだけは未調整の軸が残っている
2. 上位陣が 0.00001 差で入れ替わる状況なので、**誤差に埋もれない改善があるなら拾いたい**

In [ ]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.path.abspath("src"))
print("cwd:", os.getcwd())

## 設計

`src/05_hpo.py` は**学習コードを持たない**。既存の `<model>_preprocessing.py` を引数違いで
呼ぶだけにしてある。理由は、収束設定(lr 引き下げ + early stopping)や FE 構成を本番とズラさないため。

| 方針 | 理由 |
|---|---|
| 既存スクリプトを呼ぶだけ | 本番と同じパイプラインで測る。構成がズレると比較にならない |
| **1本ずつ順番に実行** | 8コア環境で並列にすると CPU を取り合って完走しない(実際に CatBoost が 0.33 コアまで押し出された) |
| **実行前に所要時間を見積もる** | `--estimate` で試行数 × 実測の1本あたり時間を出す |
| 判定は paired DeLong 検定 | AUC の目視では 0.0001 以下の差を判断できない |
| CatBoost は成果物を保存しない | `--save` が本番の成果物を上書きしてしまうため |

## 探索する軸

現行ベストと同じ値は含めていない(基準はスクリプト側の `BASELINE` が持っている)。

In [ ]:
import hpo_search as hpo
import pandas as pd

rows = []
for model, axes in hpo.SPACE.items():
    for axis, items in axes.items():
        rows.append([model, axis, len(items), ", ".join(n for n, _ in items)])
pd.DataFrame(rows, columns=["モデル", "軸", "試行数", "試行名"])

### 軸を選んだ理由

**LightGBM**
- `num_leaves` — **デフォルト 31 のまま**。特徴量が 92 列に増えているので、木の表現力が足りていない可能性がある。今回の本命
- `feature_fraction` / `max_depth` — 0.3 / 5 が最良かの確認。外部調査では誤差だった

**XGBoost**
- `max_depth` / `colsample_bytree` — 同上
- `min_child_weight` / `subsample` — 未調整の軸。行方向のサンプリングは列方向とは別の効き方をする可能性

**CatBoost**
- `depth` / `one_hot_max_size` — 未調整の軸
- **`rsm`(列サンプリング)は探索しない。** 実測で有害と判明済み(-0.00015〜-0.00039)。
  対称木はすべての深さで同じ分割条件を使うため、列を間引くと木全体が一斉に弱くなる

## 所要時間の見積もり

**実行するかどうかは、この見積もりを見てから判断する。**

In [ ]:
for model in hpo.SPACE:
    n = len(hpo.build_trials(model))
    per = hpo.RUNTIME[model]
    print(f"{model:9s} {n:2d} 本 × {per/60:4.1f} 分 = 約 {n*per/3600:4.1f} 時間")

| モデル | 試行数 | 1本あたり | 合計 | 優先度 |
|---|---|---|---|---|
| LightGBM | 9 | 約 4.2 分 | **約 0.6 時間** | **高**(`num_leaves` が未調整) |
| XGBoost | 10 | 約 10.8 分 | 約 1.8 時間 | 中 |
| CatBoost | 5 | 約 66.7 分 | 約 5.6 時間 | **低**(現在アンサンブルの重みが 0) |

**合計 約 8 時間。** CatBoost が全体の7割を占めるが、そのCatBoostは現在アンサンブルに
寄与していない(重み 0)。**まず LightGBM の 0.6 時間だけ回すのが費用対効果が高い。**

## 実行

```bash
uv run src/05_hpo.py lgbm --estimate           # 見積もりだけ
uv run src/05_hpo.py lgbm                      # 実行(9本、約40分)
uv run src/05_hpo.py lgbm --only num_leaves    # 本命の軸だけ(3本、約13分)
```

結果は `hpo_results.csv` に追記される。下のセルはノートブックから直接見積もりを出す例。

In [ ]:
trials = hpo.build_trials("lgbm")
hpo.estimate("lgbm", trials)

## 判定

AUC の数字を見比べるのではなく、**paired DeLong 検定**で有意性を確認する(工程⑦)。

```bash
uv run src/07_compare_oof.py lgbm --all
```

| | 従来 | DeLong |
|---|---|---|
| ノイズ床(SE) | 0.00015 | **0.00003 前後** |
| 採否基準 | 差分 ≥ +0.0002 | 差分 ≥ +0.00008 **かつ** z ≥ 3 |

**この工程を省略してはいけない。** CV で +0.000009(z=+1.57、有意でない)の構成を提出したところ、
LB では -0.00002 と逆に動いた実例がある。

## 結果

**(未実行 — 記入予定)**

実行後、以下の表を埋める。

### LightGBM

| 軸 | 試行 | OOF AUC | 基準との差 | z | p | 判定 |
|---|---|---|---|---|---|---|
| num_leaves | nl15 | | | | | |
| num_leaves | nl63 | | | | | |
| num_leaves | nl127 | | | | | |
| feature_fraction | ff02 | | | | | |
| feature_fraction | ff04 | | | | | |
| feature_fraction | ff05 | | | | | |
| max_depth | d4 | | | | | |
| max_depth | d6 | | | | | |
| max_depth | d7 | | | | | |

基準: LightGBM 現行ベスト **0.946095**

### XGBoost

**(未実行 — 記入予定)** 基準: **0.946077**

### CatBoost

**(未実行 — 記入予定)** 基準: **0.94589**

### アンサンブルへの反映

**(未実行 — 記入予定)** 単体で改善しても、アンサンブルに寄与するとは限らない。
採用された構成で `06_ensemble_hillclimb.py` を回し直し、**アンサンブル全体でも DeLong 検定を通す**。

| | CV | z | Public LB |
|---|---|---|---|
| 現行ベスト | 0.946234 | — | 0.94645 |
| HPO 後 | | | |

In [ ]:
# 実行後にこのセルで結果を確認する
hpo.report()

## まとめ(記入予定)

実行後、以下を記録する。

- どの軸が効いたか / 効かなかったか
- **なぜそうなったかの考察**(Log.md の表と同じ粒度で)
- アンサンブルに反映したか、提出したか

現時点の見立てとしては、`num_leaves` 以外は外部調査どおり誤差に終わる可能性が高い。
**「効かなかった」という結果も、重複検証を防ぐ記録として Log.md に残す。**